# 01 — Data Pull, Inspect & Validate (Sample Databases)

**Thesis:** Implied Volatility Smile Spillovers (AP-33)  
**Author:** Başar Hacımustafaoğlu — 14866196  
**Purpose:** Pull a small sample from each accessible thesis-relevant database, inspect the structure, and validate the data quality.  

---

## What this notebook does

1. Connects to WRDS
2. For each accessible database: pulls a small sample (top 500 rows)
3. Runs full validation on each pull (shape, dtypes, missingness, date coverage, duplicates)
4. Saves raw pulls to `data/raw/` as CSV
5. Saves a validation summary report to `logs/`

**This notebook does NOT clean, merge, or analyse anything.**  
**It only answers: what does this data look like, and is it usable?**

---

> ⚠️ **Run notebook 00 first** to verify your WRDS connection.

## Step 1 — Imports and connection

In [1]:
import os
import sys
import datetime
import json
import pandas as pd
import wrds
from dotenv import load_dotenv

sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
from wrds_utils import connect_wrds, validate_df

load_dotenv()
conn = connect_wrds()

TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
RAW_DIR   = "../data/raw"
LOG_DIR   = "../logs"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print(f"Run timestamp: {TIMESTAMP}")

Connecting to WRDS as: basar
Loading library list...
Done
Connection established.
Run timestamp: 20260308_154624


## Step 2 — Define pull targets

For each database we have access to, we specify which table to pull a sample from and why.

In [4]:
# Each entry: (library, table, description, n_rows_to_pull)
# We pull only 500 rows per table — this is a structural inspection, not a data download

PULL_TARGETS = [

    # OptionMetrics US sample (2014)
    ("optionmsamp_us",     "vsurfd2014",              "OptionMetrics US — volatility surface (2014)",        500),
    ("optionmsamp_us",     "opprcd2014",              "OptionMetrics US — raw option prices (2014)",         500),
    ("optionmsamp_us",     "secnmd",                  "OptionMetrics US — security names",                   500),
    ("optionmsamp_us",     "secprd",                  "OptionMetrics US — underlying prices",                500),

    # OptionMetrics Europe sample (2013)
    ("optionmsamp_europe", "volatility_surface_2013", "OptionMetrics EU — volatility surface (2013)",        500),
    ("optionmsamp_europe", "option_price_2013",       "OptionMetrics EU — raw option prices (2013)",         500),
    ("optionmsamp_europe", "security_name",           "OptionMetrics EU — security names",                   500),
    ("optionmsamp_europe", "security_price",          "OptionMetrics EU — underlying prices",                500),
    ("optionmsamp_europe", "historical_volatility",   "OptionMetrics EU — historical volatility",            500),

    # CBOE
    ("cboe",               "cboe",                    "CBOE — VIX and volatility indices",                   500),

    # FRB
    ("frb",                "rates_daily",             "FRB — daily interest rates",                          500),

    # CRSP
    ("crsp",               "dsf",                     "CRSP — daily stock file",                             500),
    ("crsp",               "dsp500",                  "CRSP — daily S&P 500 index",                          500),
    ("crsp",               "dsi",                     "CRSP — daily market index",                           500),

    # Compustat Global
    ("comp",               "g_secd",                  "Compustat Global — daily security prices",            500),
    ("comp",               "g_idx_daily",             "Compustat Global — daily index prices",               500),
    ("comp",               "g_exrt_dly",              "Compustat Global — daily exchange rates",             500),

    # Fama-French
    ("ff",                 "factors_daily",           "Fama-French — daily factors",                         500),

    # Dow Jones
    ("djones",             "djdaily",                 "Dow Jones — daily averages",                          500),

    # PHLX
    ("phlx",               "iv",                      "PHLX — FX implied volatility",                        500),

    # WRDS Apps
    ("wrdsapps",           "eushort",                 "WRDS Apps — EU short sales",                          500),
    ("wrdsapps",           "intl_market_returns",     "WRDS Apps — international market returns",            500),

    # MacroFin
    ("macrofin",           "fxdata",                  "MacroFin — FX and macro data",                        500),
    ("macrofin",           "q_factors_daily",         "MacroFin — Q-factors daily",                          500),
]

print(f"Pull targets defined: {len(PULL_TARGETS)} tables")

Pull targets defined: 24 tables


## Step 3 — Pull and validate each table

For each target we:
1. Pull `n` rows
2. Run validation
3. Save raw CSV to `data/raw/`
4. Log the result

Errors are caught and logged — a failed pull does not stop the notebook.

In [5]:
pull_log = []
pulled_dfs = {}  # keep in memory for inspection below

for library, table, description, n_rows in PULL_TARGETS:

    print(f"\n{'='*60}")
    print(f"Pulling: {library}.{table}")
    print(f"Description: {description}")
    print(f"{'='*60}")

    log_entry = {
        "library":     library,
        "table":       table,
        "description": description,
        "n_requested": n_rows,
        "status":      None,
        "n_rows":      None,
        "n_cols":      None,
        "error":       None,
        "saved_to":    None,
    }

    try:
        # Pull n rows — obs parameter limits rows returned
        df = conn.get_table(library=library, table=table, obs=n_rows)

        log_entry["status"] = "SUCCESS"
        log_entry["n_rows"] = len(df)
        log_entry["n_cols"] = len(df.columns)

        # Run validation
        validate_df(df, f"{library}.{table}")

        # Save raw CSV — filename includes library, table, and timestamp
        # Never overwrite: timestamp ensures each pull is versioned
        filename = f"{library}__{table}__{TIMESTAMP}.csv"
        filepath = os.path.join(RAW_DIR, filename)
        df.to_csv(filepath, index=False)
        log_entry["saved_to"] = filepath
        print(f"\n✓ Saved to: {filepath}")

        # Keep in memory for interactive inspection
        pulled_dfs[f"{library}.{table}"] = df

    except Exception as e:
        log_entry["status"] = "ERROR"
        log_entry["error"]  = str(e)
        print(f"\n✗ ERROR: {e}")
        print("  Skipping this table and continuing.")

    pull_log.append(log_entry)

print(f"\n{'='*60}")
print("All pull attempts complete.")


Pulling: optionmsamp_us.vsurfd2014
Description: OptionMetrics US — volatility surface (2014)

VALIDATION REPORT: optionmsamp_us.vsurfd2014

[1] Shape: 500 rows x 9 columns

[2] Columns and dtypes:
    secid                               Float64
    date                                string
    days                                Float64
    delta                               Float64
    impl_volatility                     Float64
    impl_strike                         Float64
    impl_premium                        Float64
    dispersion                          Float64
    cp_flag                             string

[3] Missingness:
    No missing values detected.

[4] Date coverage:
    date: 2014-03-03 00:00:00 → 2014-03-04 00:00:00

[5] Duplicate rows: 0



✓ Saved to: ../data/raw/optionmsamp_us__vsurfd2014__20260308_154624.csv

Pulling: optionmsamp_us.opprcd2014
Description: OptionMetrics US — raw option prices (2014)

VALIDATION REPORT: optionmsamp_us.opprcd2014

[1] Shape: 5

## Step 4 — Pull summary

In [6]:
summary_df = pd.DataFrame(pull_log)

print("\nPULL SUMMARY")
print("=" * 60)
print(summary_df[["library", "table", "status", "n_rows", "n_cols"]].to_string(index=False))

n_success = (summary_df["status"] == "SUCCESS").sum()
n_error   = (summary_df["status"] == "ERROR").sum()
print(f"\nSuccessful pulls : {n_success}")
print(f"Failed pulls     : {n_error}")

if n_error > 0:
    print("\nFailed tables:")
    failed = summary_df[summary_df["status"] == "ERROR"]
    for _, row in failed.iterrows():
        print(f"  {row['library']}.{row['table']}: {row['error']}")


PULL SUMMARY
           library                   table  status  n_rows  n_cols
    optionmsamp_us              vsurfd2014 SUCCESS   500.0     9.0
    optionmsamp_us              opprcd2014 SUCCESS   500.0    22.0
    optionmsamp_us                  secnmd SUCCESS     3.0     8.0
    optionmsamp_us                  secprd SUCCESS    10.0    11.0
optionmsamp_europe volatility_surface_2013 SUCCESS   500.0    10.0
optionmsamp_europe       option_price_2013 SUCCESS   500.0    24.0
optionmsamp_europe           security_name SUCCESS     1.0     6.0
optionmsamp_europe          security_price SUCCESS    50.0    14.0
optionmsamp_europe   historical_volatility SUCCESS   143.0     5.0
              cboe                    cboe SUCCESS   500.0    17.0
               frb             rates_daily SUCCESS   500.0    83.0
              crsp                     dsf SUCCESS   500.0    20.0
              crsp                  dsp500 SUCCESS   500.0    11.0
              crsp                     dsi SUCCE

## Step 5 — Interactive inspection

Use the cells below to look at specific tables in more detail.  
The most important ones for your thesis are the volatility surface tables.

In [20]:
import pandas as pd

# ================================================================
# STEP 5 — INTERACTIVE INSPECTION
# Run this single cell to inspect all thesis-critical tables
# ================================================================

def inspect(key, extra_cols=None):
    if key not in pulled_dfs:
        print(f"✗ '{key}' not available.\n")
        return
    df = pulled_dfs[key]
    print(f"{'='*60}")
    print(f"TABLE: {key}")
    print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"Columns: {list(df.columns)}")
    print(f"\nFirst 5 rows (transposed):")
    print(df.head(5).T.to_string())
    print()

# ----------------------------------------------------------------
# 1. US Volatility Surface — PRIMARY THESIS TABLE
# ----------------------------------------------------------------
inspect("optionmsamp_us.vsurfd2014")

# ----------------------------------------------------------------
# 2. EU Volatility Surface — PRIMARY THESIS TABLE
# ----------------------------------------------------------------
inspect("optionmsamp_europe.volatility_surface_2013")

# ----------------------------------------------------------------
# 3. US Security Names — find SPX secid
# ----------------------------------------------------------------
inspect("optionmsamp_us.secnmd")

# ----------------------------------------------------------------
# 4. EU Security Names — find Euro Stoxx 50 secid
# ----------------------------------------------------------------
inspect("optionmsamp_europe.security_name")

# ----------------------------------------------------------------
# 5. US Underlying Prices
# ----------------------------------------------------------------
inspect("optionmsamp_us.secprd")

# ----------------------------------------------------------------
# 6. EU Underlying Prices
# ----------------------------------------------------------------
inspect("optionmsamp_europe.security_price")

# ----------------------------------------------------------------
# 7. CBOE VIX
# ----------------------------------------------------------------
inspect("cboe.cboe")

# ----------------------------------------------------------------
# 8. FRB Daily Rates — show only thesis-relevant columns
# ----------------------------------------------------------------
if "frb.rates_daily" in pulled_dfs:
    df = pulled_dfs["frb.rates_daily"]
    print("="*60)
    print("TABLE: frb.rates_daily")
    print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
    cols = ["date", "dff", "dtb3", "dgs1", "dgs2", "dgs10", "sofr", "t10y2y", "tedrate", "bamlh0a0hym2"]
    available = [c for c in cols if c in df.columns]
    print(f"Thesis-relevant columns available: {available}")
    print(f"\nRecent rows (last 5, transposed):")
    print(df[available].dropna(subset=["dff"]).tail(5).T.to_string())
    print()

# ----------------------------------------------------------------
# 9. CRSP Daily Index (dsi) and DSF
# ----------------------------------------------------------------
inspect("crsp.dsi")
inspect("crsp.dsf")

# ----------------------------------------------------------------
# 10. Compustat Global Daily Index
# ----------------------------------------------------------------
inspect("comp.g_idx_daily")

# ----------------------------------------------------------------
# 11. Compustat Global Exchange Rates
# ----------------------------------------------------------------
inspect("comp.g_exrt_dly")

# ----------------------------------------------------------------
# 12. Fama-French Daily Factors
# ----------------------------------------------------------------
inspect("ff.factors_daily")

# ----------------------------------------------------------------
# 13. Dow Jones Daily
# ----------------------------------------------------------------
inspect("djones.djdaily")

# ----------------------------------------------------------------
# 14. PHLX FX Implied Volatility
# ----------------------------------------------------------------
inspect("phlx.iv")

# ----------------------------------------------------------------
# 15. WRDS Apps — EU Short Selling
# ----------------------------------------------------------------
inspect("wrdsapps.eushort")

# ----------------------------------------------------------------
# 16. MacroFin FX Data
# ----------------------------------------------------------------
inspect("macrofin.fxdata")

# ----------------------------------------------------------------
# 17. MacroFin Q-Factors Daily
# ----------------------------------------------------------------
inspect("macrofin.q_factors_daily")

print("="*60)
print("Step 5 complete.")

TABLE: optionmsamp_us.vsurfd2014
Shape: 500 rows x 9 columns
Columns: ['secid', 'date', 'days', 'delta', 'impl_volatility', 'impl_strike', 'impl_premium', 'dispersion', 'cp_flag']

First 5 rows (transposed):
                          0           1           2           3           4
secid              101594.0    101594.0    101594.0    101594.0    101594.0
date             2014-03-03  2014-03-03  2014-03-03  2014-03-03  2014-03-03
days                   30.0        30.0        30.0        30.0        30.0
delta                 -80.0       -75.0       -70.0       -65.0       -60.0
impl_volatility    0.230443    0.221934    0.217388     0.21489    0.213312
impl_strike        559.1971    552.0703    546.4105    541.5253    537.0652
impl_premium       35.15058    29.10125    24.64949    21.10779     18.1251
dispersion         0.028626    0.014204    0.007244    0.004472      0.0033
cp_flag                   P           P           P           P           P

TABLE: optionmsamp_europe.volat

## Step 6 — Save full log

In [21]:
log_path = os.path.join(LOG_DIR, f"data_pull_log_{TIMESTAMP}.json")

with open(log_path, "w") as f:
    json.dump(pull_log, f, indent=2, default=str)

print(f"Pull log saved to: {log_path}")

Pull log saved to: ../logs/data_pull_log_20260308_154624.json


## Step 7 — Close connection

In [22]:
conn.close()
print("WRDS connection closed.")
print("\n✓ Notebook 01 complete.")
print("\nNext steps:")
print("  - Review the validation reports above")
print("  - Check data/raw/ for saved CSVs")
print("  - Check logs/ for the pull log")
print("  - Note any tables that failed or returned unexpected structures")
print("  - Report findings before building notebook 02 (cleaning)")

WRDS connection closed.

✓ Notebook 01 complete.

Next steps:
  - Review the validation reports above
  - Check data/raw/ for saved CSVs
  - Check logs/ for the pull log
  - Note any tables that failed or returned unexpected structures
  - Report findings before building notebook 02 (cleaning)
